# Phase I · Error vs. speed: binned summaries

**Question:** does the Translation model's (`transnet18`) velocity error depend on how fast the platform moves? And do any conclusions depend on frames 546–559, the suspected ground-truth (GT) anomaly from `01_velocity_timeseries`?

Frames are grouped into speed bins (default 0.05 m/s). Each bin is summarised for three errors:

| Row | Error | Meaning |
|---|---|---|
| Forward | eₓ = predicted vx − true vx | + = model says faster forward than GT |
| Lateral | e_y = predicted vy − true vy | + = predicted vy above true vy (the sign convention of y is not yet verified) |
| Absolute | ‖e‖ = √(eₓ² + e_y²) | 2-D vector error, m/s |

Each bin is drawn twice, side by side:
- **All frames**
- **Excluding frames 546–559**

Frames 546–559 are left out only as a **sensitivity check**. Whether that GT is wrong is *not* settled. If a pattern looks the same in both versions, it does not depend on those frames.

**How to read a bin:** the dot is the median frame, the thick bar spans the middle 50% of frames (IQR), the thin line spans 10–90%, and the diamond is the mean. A **hollow dot** means the bin has fewer than 30 frames or fewer than 3 separate time windows, so read it with care. Consecutive frames are not independent, so the window count is a better guide to how much evidence a bin holds than the frame count.

**Interaction:** click a bin to see *when* its frames occur (bottom strip); double-click to clear. Click a legend entry to emphasise one version. The *Speed axis* menu (below the chart) chooses what defines a frame's speed bin.

### Why three speed axes?
Binning frames by a **noisy** quantity creates a trend by itself (regression to the mean). Frames whose GT speed is unusually high are partly frames where GT noise happened to be positive, so pred − GT looks negative there, even for a perfect model.
- **True speed (GT):** the standard choice; carries GT's frame-to-frame noise.
- **True speed, 0.5 s smoothed:** centred 15-frame mean of GT vx and vy before taking the speed. This removes most frame-level GT noise from the binning (frames 546–559 are masked before smoothing in the "excluding" version).
- **Predicted speed:** bins by what the model says. Here noise in the *prediction* would push the artefact the other way.

A trend that keeps its shape across the axes is more likely a property of the model. One that flips or vanishes points to noise in the binning variable.

**Requires:** `altair >= 5.4` (Altair 6.0 for Python 3.9). Optional: `vl-convert-python`, which embeds the chart's JavaScript so the saved HTML also works offline.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

try:
    import altair as alt
except ImportError as e:
    raise ImportError("Altair is missing. In the RadaVis environment run:  pip install altair") from e
if not hasattr(alt, "when"):
    raise ImportError(f"Altair {alt.__version__} is too old for this notebook; run:  pip install -U altair")

print("altair", alt.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

altair 6.0.0 | pandas 2.2.3 | numpy 1.26.4


## 1 · Configuration

In [2]:
SEQUENCE = "bot_beckman_5thfloor_1"
SPLIT = "main_0"                 # checkpoint that produced the predictions (output_main_0/transnet18)
SUSPECT_FRAMES = (546, 559)      # inclusive; suspected GT anomaly (see 01_velocity_timeseries)

BIN_WIDTH = 0.05                 # m/s
SMOOTH_FRAMES = 15               # centred window for the smoothed-GT speed axis (~0.5 s at 30 Hz)
MIN_FRAMES, MIN_WINDOWS = 30, 3  # below either: bin drawn hollow ("few frames / few separate moments")
WINDOW_GAP_S = 1.0               # frames more than 1 s apart count as separate time windows


def find_repo_root(start: Path) -> Path:
    # Walk up from the notebook's folder to the RadaVis root (the folder holding results/ and notebooks/).
    for p in [start, *start.parents]:
        if (p / "results").is_dir() and (p / "notebooks").is_dir():
            return p
    raise FileNotFoundError("RadaVis root not found above %s" % start)


ROOT = find_repo_root(Path.cwd().resolve())
NPZ_PATH = ROOT / "results" / "phase1" / SPLIT / f"{SEQUENCE}_predictions.npz"
HTML_OUT = NPZ_PATH.with_name(f"{SEQUENCE}_error_vs_speed.html")
SPLIT_CONFIG = ROOT / "radarize_ae" / "configs" / f"{SPLIT}.yaml"   # optional: used for the held-out check

print("RadaVis root:", ROOT)
print("Predictions :", NPZ_PATH)

RadaVis root: F:\visualization\RadaVis
Predictions : F:\visualization\RadaVis\results\phase1\main_0\bot_beckman_5thfloor_1_predictions.npz


## 2 · Load and sanity-check the predictions
Same checks as in `01_velocity_timeseries`, plus a check of what frames 546–559 look like in the ground truth.

In [3]:
REQUIRED = ["time", "flow_pred_xs", "flow_pred_ys", "flow_gt_xs", "flow_gt_ys"]

with np.load(NPZ_PATH, allow_pickle=False) as npz:
    missing = [k for k in REQUIRED if k not in npz.files]
    if missing:
        raise KeyError(f"{NPZ_PATH.name} lacks {missing}; found {npz.files}")
    raw = {k: np.asarray(npz[k], dtype=np.float64).ravel() for k in REQUIRED}

n = len(raw["time"])
assert len({len(v) for v in raw.values()}) == 1, "arrays differ in length"
assert all(np.isfinite(v).all() for v in raw.values()), "NaN/inf in the predictions file"
assert (np.diff(raw["time"]) > 0).all(), "time is not strictly increasing"
print(f"frames: {n}, duration {raw['time'][-1] - raw['time'][0]:.2f} s")


def held_out_sequences(cfg_path: Path):
    # Read TEST_SPLIT from a Radarize split config without needing PyYAML.
    seqs, in_test = [], False
    for line in cfg_path.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith("TEST_SPLIT:"):
            in_test = True
            continue
        if in_test:
            if stripped.startswith("- "):
                seqs.append(stripped[2:].strip())
            elif stripped:
                break
    return seqs


if SPLIT_CONFIG.exists():
    ok = SEQUENCE in held_out_sequences(SPLIT_CONFIG)
    print("held-out check:", "OK" if ok else "WARNING: sequence was seen in training/validation by this checkpoint")
else:
    print(f"held-out check: skipped ({SPLIT_CONFIG} not found)")

df = pd.DataFrame({
    "frame": np.arange(n),
    "t": raw["time"] - raw["time"][0],
    "gt_vx": raw["flow_gt_xs"], "gt_vy": raw["flow_gt_ys"],
    "pred_vx": raw["flow_pred_xs"], "pred_vy": raw["flow_pred_ys"],
})
df["err_x"] = df["pred_vx"] - df["gt_vx"]
df["err_y"] = df["pred_vy"] - df["gt_vy"]
df["err_norm"] = np.hypot(df["err_x"], df["err_y"])
df["suspect"] = df["frame"].between(*SUSPECT_FRAMES)

# How unusual is the suspected window? Runs of exactly repeated GT values:
same_as_prev = (df["gt_vx"].diff() == 0) & (df["gt_vy"].diff() == 0)
run_lengths = df.groupby((~same_as_prev).cumsum())["frame"].agg(["first", "count"])
print("GT runs of identical values, length -> how often:", run_lengths["count"].value_counts().sort_index().to_dict())
longest = run_lengths.loc[run_lengths["count"].idxmax()]
print(f"longest run: {int(longest['count'])} frames starting at frame {int(longest['first'])}")

frames: 1798, duration 59.90 s
held-out check: OK
GT runs of identical values, length -> how often: {1: 826, 2: 479, 14: 1}
longest run: 14 frames starting at frame 546


## 3 · Speed axes, bins and per-bin summaries

In [4]:
VARIANTS = {"all": "All frames", "excl": f"Excluding frames {SUSPECT_FRAMES[0]}–{SUSPECT_FRAMES[1]}"}
BASES = {"gt": "true speed (GT)", "gt_smooth": "true speed, 0.5 s smoothed (GT)", "pred": "predicted speed"}
METRICS = {"err_x": "forward error", "err_y": "lateral error", "err_norm": "absolute error"}


def smoothed_gt_speed(mask=None):
    vx, vy = df["gt_vx"], df["gt_vy"]
    if mask is not None:                       # treat masked GT values as missing, not as zero
        vx, vy = vx.mask(mask), vy.mask(mask)
    kw = dict(window=SMOOTH_FRAMES, center=True, min_periods=SMOOTH_FRAMES // 3)
    return np.hypot(vx.rolling(**kw).mean(), vy.rolling(**kw).mean())


frame_tables = []
for variant in VARIANTS:
    speeds = pd.DataFrame({
        "gt": np.hypot(df["gt_vx"], df["gt_vy"]),
        "gt_smooth": smoothed_gt_speed(df["suspect"] if variant == "excl" else None),
        "pred": np.hypot(df["pred_vx"], df["pred_vy"]),
    })
    keep = ~df["suspect"] if variant == "excl" else pd.Series(True, index=df.index)
    ft = pd.concat([df, speeds.add_prefix("speed_")], axis=1)[keep].copy()
    ft["variant"], ft["variant_label"] = variant, VARIANTS[variant]
    frame_tables.append(ft)
frames_long = pd.concat(frame_tables, ignore_index=True)

top = np.ceil(frames_long[[f"speed_{b}" for b in BASES]].max().max() / BIN_WIDTH) * BIN_WIDTH
EDGES = np.round(np.arange(0, top + BIN_WIDTH / 2, BIN_WIDTH), 4)
BIN_LABELS = [f"{lo:.2f}–{hi:.2f}" for lo, hi in zip(EDGES[:-1], EDGES[1:])]
for b in BASES:
    frames_long[f"bin_{b}"] = pd.cut(frames_long[f"speed_{b}"], EDGES, right=False, labels=BIN_LABELS).astype(str)


def time_windows(times) -> int:
    t = np.sort(np.asarray(times))
    return int(1 + np.sum(np.diff(t) > WINDOW_GAP_S)) if len(t) else 0


def summarise_bin(g, metric):
    v = g[metric]
    q = v.quantile([0.10, 0.25, 0.50, 0.75, 0.90]).to_numpy()
    return {"n": len(v), "windows": time_windows(g["t"]),
            "q10": q[0], "q25": q[1], "q50": q[2], "q75": q[3], "q90": q[4],
            "mean": v.mean(), "rmse": float(np.sqrt((v ** 2).mean()))}


rows = []
for variant in VARIANTS:
    sub = frames_long[frames_long["variant"] == variant]
    for basis in BASES:
        for label, g in sub.groupby(f"bin_{basis}"):
            if label == "nan":                  # smoothed speed undefined (masked frames)
                continue
            for metric in METRICS:
                rows.append({"variant": variant, "variant_label": VARIANTS[variant], "basis": basis,
                             "metric": metric, "bin_label": label, **summarise_bin(g, metric)})
summary = pd.DataFrame(rows)
summary["low_support"] = (summary["n"] < MIN_FRAMES) | (summary["windows"] < MIN_WINDOWS)

# Frames per bin, split into "other frames" and the suspected window (all-frames version)
all_frames = frames_long[frames_long["variant"] == "all"]
count_rows = []
for basis in BASES:
    c = all_frames.groupby([f"bin_{basis}", "suspect"]).size()
    for (label, is_suspect), k in c.items():
        if label != "nan":
            count_rows.append({"basis": basis, "bin_label": label, "count": int(k),
                               "part": "suspect" if is_suspect else "other"})
counts = pd.DataFrame(count_rows)
totals = counts.groupby(["basis", "bin_label"], as_index=False)["count"].sum()

# Only the columns the chart needs (keeps the embedded data small)
chart_frames = frames_long[["frame", "t", "variant", "variant_label", "err_x", "err_norm"]
                           + [f"speed_{b}" for b in BASES] + [f"bin_{b}" for b in BASES]].round(4)
print(f"{len(BIN_LABELS)} bins of {BIN_WIDTH} m/s: {BIN_LABELS[0]} … {BIN_LABELS[-1]}")

9 bins of 0.05 m/s: 0.00–0.05 … 0.40–0.45


### Table view (true-speed bins)
Medians and means per bin, for both versions; `Δ` columns show what removing frames 546–559 changes.

In [5]:
def bin_table(basis="gt"):
    s = summary[summary["basis"] == basis]
    wide = s.pivot_table(index="bin_label", columns=["metric", "variant"],
                         values=["n", "q50", "mean", "rmse"], aggfunc="first")
    out = pd.DataFrame({
        "frames (all)": wide[("n", "err_x", "all")],
        "frames (excl.)": wide[("n", "err_x", "excl")],
        "fwd median": wide[("q50", "err_x", "all")],
        "fwd mean": wide[("mean", "err_x", "all")],
        "Δ fwd mean": wide[("mean", "err_x", "excl")] - wide[("mean", "err_x", "all")],
        "|e| median": wide[("q50", "err_norm", "all")],
        "|e| mean": wide[("mean", "err_norm", "all")],
        "Δ |e| mean": wide[("mean", "err_norm", "excl")] - wide[("mean", "err_norm", "all")],
        "RMSE": wide[("rmse", "err_norm", "all")],
        "Δ RMSE": wide[("rmse", "err_norm", "excl")] - wide[("rmse", "err_norm", "all")],
    }).reindex([b for b in BIN_LABELS if b in wide.index])
    return out


display(bin_table("gt").round(3))

,frames (all),frames (excl.),fwd median,fwd mean,Δ fwd mean,|e| median,|e| mean,Δ |e| mean,RMSE,Δ RMSE
bin_label,,,,,,,,,,
0.00–0.05,46,46,0.006,0.024,0.00,0.031,0.048,0.000,0.072,0.000
0.05–0.10,144,144,-0.010,-0.002,0.00,0.051,0.062,0.000,0.074,0.000
0.10–0.15,74,74,0.031,0.040,0.00,0.082,0.083,0.000,0.093,0.000
0.15–0.20,216,202,0.075,0.080,-0.02,0.081,0.099,-0.021,0.130,-0.048
0.20–0.25,476,476,0.024,0.025,0.00,0.037,0.039,0.000,0.046,0.000
0.25–0.30,517,517,-0.022,-0.023,0.00,0.038,0.041,0.000,0.047,0.000
0.30–0.35,278,278,-0.068,-0.070,0.00,0.078,0.078,0.000,0.081,0.000
0.35–0.40,44,44,-0.120,-0.119,0.00,0.130,0.128,0.000,0.129,0.000
0.40–0.45,3,3,-0.155,-0.154,0.00,0.174,0.174,0.000,0.174,0.000


## 4 · Interactive chart

In [6]:
# Colours: two categorical slots (CVD-checked); greys for structure; text in ink, never series colours.
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
VARIANT_COLORS = ["#2a78d6", "#eb6834"]          # all frames, excluding suspected frames
COUNT_OTHER, COUNT_SUSPECT = "#b4b2a7", INK_2
WIDTH = 760
VARIANT_LABELS = list(VARIANTS.values())
SUSPECT_LABEL = f"frames {SUSPECT_FRAMES[0]}–{SUSPECT_FRAMES[1]}"

signed = summary[summary["metric"].isin(["err_x", "err_y"])]
_m = float(np.nanmax(np.abs(signed[["q10", "q90", "mean"]].to_numpy()))) * 1.06
SIGNED_DOMAIN = [-_m, _m]                       # same scale for forward and lateral error, all speed axes
ABS_DOMAIN = [0.0, float(summary.loc[summary["metric"] == "err_norm", ["q90", "mean"]].max().max()) * 1.08]

_variant_scale = alt.Scale(domain=VARIANT_LABELS, range=VARIANT_COLORS)


def variant_color():
    # One shared colour scale -> one legend (its look is set in configure_legend below).
    return alt.Color("variant_label:N", title=None, sort=VARIANT_LABELS, scale=_variant_scale)


def x_bins(labels=False):
    axis = (alt.Axis(labels=True, labelAngle=0, labelColor=INK_2, labelFontSize=10, ticks=False, domain=False,
                     grid=False, title="speed bin (m/s)") if labels
            else alt.Axis(labels=False, ticks=False, domain=False, grid=False, title=None))
    return alt.X("bin_label:N", sort=BIN_LABELS, axis=axis,
                 scale=alt.Scale(domain=BIN_LABELS, paddingInner=0.18, paddingOuter=0.08))


X_OFFSET = alt.XOffset("variant_label:N", sort=VARIANT_LABELS, scale=alt.Scale(domain=VARIANT_LABELS, paddingInner=0.2))


def bin_tooltip(metric):
    fmt = ".3f" if metric == "err_norm" else "+.3f"
    tips = [alt.Tooltip("variant_label:N", title="version"),
            alt.Tooltip("bin_label:N", title="speed bin (m/s)"),
            alt.Tooltip("n:Q", title="frames", format="d"),
            alt.Tooltip("windows:Q", title="separate time windows", format="d"),
            alt.Tooltip("q50:Q", title="median", format=fmt),
            alt.Tooltip("q25:Q", title="25th percentile", format=fmt),
            alt.Tooltip("q75:Q", title="75th percentile", format=fmt),
            alt.Tooltip("q10:Q", title="10th percentile", format=fmt),
            alt.Tooltip("q90:Q", title="90th percentile", format=fmt),
            alt.Tooltip("mean:Q", title="mean", format=fmt)]
    if metric == "err_norm":
        tips.append(alt.Tooltip("rmse:Q", title="RMSE", format=".3f"))
    return tips


def make_error_vs_speed_chart(title: str) -> alt.VConcatChart:
    # Interactions ------------------------------------------------------------------
    basis = alt.param(name="basis", value="gt", bind=alt.binding_select(
        options=list(BASES), labels=list(BASES.values()), name="Speed axis: "))
    variant_pick = alt.selection_point(name="variant_pick", fields=["variant_label"], bind="legend")
    bin_pick = alt.selection_point(name="bin_pick", fields=["variant", "bin_label"], on="click", clear="dblclick")
    lit = bin_pick & variant_pick                  # both empty -> everything is drawn at full strength

    def summary_row(metric, y_title, domain, with_legend=False, height=125):
        base = alt.Chart(summary).transform_filter(alt.datum.basis == basis).transform_filter(
            alt.datum.metric == metric)

        def y(field):
            return alt.Y(f"{field}:Q", title=y_title, scale=alt.Scale(domain=domain, nice=False, zero=False),
                         axis=alt.Axis(tickCount=5, format=".2f"))

        enc = dict(x=x_bins(), xOffset=X_OFFSET)
        whisker = base.mark_rule(strokeWidth=1.5).encode(
            **enc, y=y("q10"), y2="q90:Q",
            color=variant_color(),
            opacity=alt.when(lit).then(alt.value(1.0)).otherwise(alt.value(0.25)))
        if with_legend:
            whisker = whisker.add_params(variant_pick)   # legend clicks live on the view that owns the legend
        layers = [
            whisker,
            base.mark_rule(strokeWidth=7, strokeCap="round").encode(
                **enc, y=y("q25"), y2="q75:Q", color=variant_color(),
                opacity=alt.when(lit).then(alt.value(0.45)).otherwise(alt.value(0.12))),
            base.mark_point(shape="diamond", size=42, filled=False, strokeWidth=1.5).encode(
                **enc, y=y("mean"), color=variant_color(),
                opacity=alt.when(lit).then(alt.value(1.0)).otherwise(alt.value(0.25))),
            # medians: filled dot with a surface ring; hollow dot where the bin has little support
            base.transform_filter("!datum.low_support").mark_point(
                shape="circle", size=58, filled=True, stroke=SURFACE, strokeWidth=1.5).encode(
                **enc, y=y("q50"), color=variant_color(),
                opacity=alt.when(lit).then(alt.value(1.0)).otherwise(alt.value(0.25))),
            base.transform_filter("datum.low_support").mark_point(
                shape="circle", size=58, filled=False, fill=SURFACE, strokeWidth=1.8).encode(
                **enc, y=y("q50"), color=variant_color(),
                opacity=alt.when(lit).then(alt.value(1.0)).otherwise(alt.value(0.25))),
            # invisible full-height target per bin and version: hover for numbers, click to select
            base.mark_bar(opacity=0).encode(**enc, y=alt.datum(domain[0]), y2=alt.datum(domain[1]),
                                            tooltip=bin_tooltip(metric)).add_params(bin_pick),
        ]
        if metric != "err_norm":
            layers.insert(0, alt.Chart(summary).mark_rule(color=AXIS, strokeWidth=1).encode(y=alt.datum(0)))
        return alt.layer(*layers).properties(width=WIDTH, height=height)

    basis_note = alt.Chart(pd.DataFrame({"k": [0]})).transform_calculate(
        label="'Speed bins from: ' + (basis == 'gt' ? 'true speed (GT), frame by frame' : "
              "basis == 'gt_smooth' ? 'true speed (GT), smoothed over 0.5 s' : 'predicted speed')"
    ).mark_text(align="left", baseline="top", fontSize=12, fontWeight="bold", color=INK).encode(
        text="label:N", x=alt.value(0), y=alt.value(0)).properties(width=WIDTH, height=14)

    counts_row = alt.layer(
        alt.Chart(counts).transform_filter(alt.datum.basis == basis).mark_bar(
            cornerRadiusTopLeft=2, cornerRadiusTopRight=2).encode(
            x=x_bins(labels=True),
            y=alt.Y("count:Q", stack="zero", title="frames", axis=alt.Axis(tickCount=3)),
            order=alt.Order("part:N", sort="ascending"),
            color=alt.when(alt.datum.part == "suspect").then(alt.value(COUNT_SUSPECT)).otherwise(alt.value(COUNT_OTHER)),
            tooltip=[alt.Tooltip("bin_label:N", title="speed bin (m/s)"),
                     alt.Tooltip("part:N", title="frames (other / suspect)"), alt.Tooltip("count:Q", title="count")]),
        alt.Chart(totals).transform_filter(alt.datum.basis == basis).mark_text(
            dy=-6, fontSize=10, color=INK_2).encode(
            x=x_bins(labels=True), y=alt.Y("count:Q", stack=None, title="frames"), text="count:Q"),
    ).properties(width=WIDTH, height=70, title=alt.TitleParams(
        f"Frames per bin, all frames (dark part: {SUSPECT_LABEL})", fontSize=11, fontWeight="normal",
        color=INK_2, anchor="start"))

    # When do the selected bin's frames occur? ----------------------------------------
    frames_view = alt.Chart(chart_frames).transform_calculate(
        speed="basis == 'gt' ? datum.speed_gt : basis == 'gt_smooth' ? datum.speed_gt_smooth : datum.speed_pred",
        bin_label="basis == 'gt' ? datum.bin_gt : basis == 'gt_smooth' ? datum.bin_gt_smooth : datum.bin_pred",
    )
    t_sus = df.loc[df["suspect"], "t"]
    half_dt = float(np.median(np.diff(df["t"]))) / 2
    window_df = pd.DataFrame({"t0": [t_sus.min() - half_dt], "t1": [t_sus.max() + half_dt], "label": [SUSPECT_LABEL]})
    T_TITLE, S_TITLE = "time since first frame (s)", "speed (m/s)"
    x_time = dict(scale=alt.Scale(domain=[0, float(df["t"].max())], nice=False), axis=alt.Axis(tickCount=12),
                  title=T_TITLE)
    strip = alt.layer(
        alt.Chart(window_df).mark_rect(color=GRID, opacity=0.9).encode(
            x=alt.X("t0:Q", **x_time), x2="t1:Q"),
        alt.Chart(window_df).mark_text(align="left", baseline="top", dx=3, dy=2, fontSize=10, color=INK_2).encode(
            x=alt.X("t1:Q", **x_time), y=alt.value(0), text="label:N"),
        frames_view.transform_filter(alt.datum.variant == "all").mark_line(color=MUTED, strokeWidth=1).encode(
            x=alt.X("t:Q", **x_time), y=alt.Y("speed:Q", title=S_TITLE, axis=alt.Axis(tickCount=3),
                                              scale=alt.Scale(domain=[0, float(EDGES[-1])], nice=False))),
        frames_view.transform_filter({"param": "bin_pick", "empty": False}).mark_circle(size=16, opacity=0.9).encode(
            x=alt.X("t:Q", **x_time), y=alt.Y("speed:Q", title=S_TITLE,
                                              scale=alt.Scale(domain=[0, float(EDGES[-1])], nice=False)),
            color=variant_color(),
            tooltip=[alt.Tooltip("frame:Q", title="frame"), alt.Tooltip("t:Q", title="t (s)", format=".2f"),
                     alt.Tooltip("speed:Q", title="speed (m/s)", format=".3f"),
                     alt.Tooltip("err_x:Q", title="forward error", format="+.3f"),
                     alt.Tooltip("err_norm:Q", title="absolute error", format=".3f")]),
    ).properties(width=WIDTH, height=95, title=alt.TitleParams(
        "When do the selected bin's frames occur? Click a bin above (line: speed on the chosen axis)",
        fontSize=11, fontWeight="normal", color=INK_2, anchor="start"))

    with warnings.catch_warnings():
        # The same click selection sits on all three rows on purpose; Altair 6 notes that it merged them.
        warnings.filterwarnings("ignore", message="Automatically deduplicated selection parameter")
        chart = alt.vconcat(
            basis_note,
            summary_row("err_x", "forward error (m/s)", SIGNED_DOMAIN, with_legend=True),
            summary_row("err_y", "lateral error (m/s)", SIGNED_DOMAIN),
            summary_row("err_norm", "absolute error (m/s)", ABS_DOMAIN),
            counts_row,
            strip,
            spacing=10,
        ).add_params(basis).properties(
            autosize=alt.AutoSizeParams(type="pad", resize=True),
            title=alt.TitleParams(title, anchor="start", fontSize=14, color=INK, subtitleColor=INK_2, subtitle=[
                "Each bin: ● median · thick bar = middle 50% of frames · thin line = 10–90% · ◇ mean · "
                "○ (hollow) = fewer than 30 frames or 3 separate time windows",
                "Error = predicted − true. Click a bin to see when its frames occur (double-click clears) · "
                "click the legend to emphasise a version · choose the speed axis below the chart",
            ]),
        ).configure(background=SURFACE, font="system-ui, -apple-system, Segoe UI, sans-serif").configure_view(
            stroke=None).configure_axis(
            gridColor=GRID, domainColor=AXIS, tickColor=AXIS, labelColor=INK_2, titleColor=INK_2,
            labelFontSize=10, titleFontSize=11, titleFontWeight="normal").configure_legend(
            orient="top", direction="horizontal", symbolType="circle", labelFontSize=11, labelColor=INK_2)
    return chart


chart = make_error_vs_speed_chart(f"{SEQUENCE}: error vs. speed (transnet18, {SPLIT}), {n} frames")
chart

alt.VConcatChart(...)

## 5 · Save the interactive HTML
Writes the chart next to the predictions. With `vl-convert-python` installed, the Vega JavaScript is embedded and the file also works offline. Otherwise it loads Vega from a CDN and needs internet to display.

In [7]:
try:
    import vl_convert  # noqa: F401  (optional)
    chart.save(str(HTML_OUT), inline=True)
    note = "self-contained, works offline"
except ImportError:
    chart.save(str(HTML_OUT))
    note = "loads Vega from a CDN, needs internet"

# Page styling: keep the 'Speed axis' menu below the chart (Altair's page puts them in a flex row)
# and give it the chart's font. Inserted right after the opening <head> tag, because the embedded
# Vega JavaScript itself contains the text "</head>".
css = ("<style>body{background:#fcfcfb;margin:16px}"
       "#vis.vega-embed{flex-direction:column;align-items:flex-start}"
       ".vega-bindings{font:13px system-ui,-apple-system,'Segoe UI',sans-serif;color:#52514e;margin:8px 0 0 4px}"
       ".vega-bindings select{font:inherit}</style>")
html = HTML_OUT.read_text(encoding="utf-8")
head = html.find("<head>")
if head != -1:
    pos = head + len("<head>")
    HTML_OUT.write_text(html[:pos] + css + html[pos:], encoding="utf-8")
print(f"saved ({note}):", HTML_OUT)

saved (loads Vega from a CDN, needs internet): F:\visualization\RadaVis\results\phase1\main_0\bot_beckman_5thfloor_1_error_vs_speed.html


In [8]:
import numpy as np
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        candidate = p / "results" / "phase1" / "main_0" / "bot_beckman_5thfloor_1_predictions.npz"
        if candidate.exists():
            return p
    raise FileNotFoundError("Could not find repo root containing predictions file.")

ROOT = find_repo_root()
PRED_PATH = ROOT / "results" / "phase1" / "main_0" / "bot_beckman_5thfloor_1_predictions.npz"

print("Repo root:", ROOT)
print("Predictions file:", PRED_PATH)

with np.load(PRED_PATH) as data:
    keys = list(data.files)
    t = data["time"]
    pred_x = data["flow_pred_xs"]
    pred_y = data["flow_pred_ys"]
    gt_x = data["flow_gt_xs"]
    gt_y = data["flow_gt_ys"]

print("Keys:", keys)
print("Lengths:", len(t), len(pred_x), len(pred_y), len(gt_x), len(gt_y))
print("Finite:",
      np.isfinite(t).all(),
      np.isfinite(pred_x).all(),
      np.isfinite(pred_y).all(),
      np.isfinite(gt_x).all(),
      np.isfinite(gt_y).all())

Repo root: f:\visualization\RadaVis
Predictions file: f:\visualization\RadaVis\results\phase1\main_0\bot_beckman_5thfloor_1_predictions.npz
Keys: ['time', 'flow_pred_xs', 'flow_pred_ys', 'flow_gt_xs', 'flow_gt_ys']
Lengths: 1798 1798 1798 1798 1798
Finite: True True True True True


In [9]:
err_x = pred_x - gt_x
err_y = pred_y - gt_y
err_abs = np.sqrt(err_x**2 + err_y**2)

gt_speed = np.sqrt(gt_x**2 + gt_y**2)
pred_speed = np.sqrt(pred_x**2 + pred_y**2)

print("Shapes:")
print("err_x:", err_x.shape)
print("err_y:", err_y.shape)
print("err_abs:", err_abs.shape)
print("gt_speed:", gt_speed.shape)
print("pred_speed:", pred_speed.shape)

print("\nBasic stats:")
print("Forward MAE:", np.mean(np.abs(err_x)))
print("Lateral MAE:", np.mean(np.abs(err_y)))
print("Absolute mean:", np.mean(err_abs))
print("Absolute median:", np.median(err_abs))

Shapes:
err_x: (1798,)
err_y: (1798,)
err_abs: (1798,)
gt_speed: (1798,)
pred_speed: (1798,)

Basic stats:
Forward MAE: 0.04838061950754551
Lateral MAE: 0.02497909579177992
Absolute mean: 0.05915018226237179
Absolute median: 0.050576622937280824


In [10]:
exclude_start = 546
exclude_end = 559   # inclusive

mask_excl = np.ones(len(t), dtype=bool)
mask_excl[exclude_start:exclude_end+1] = False

print("Total frames:", len(t))
print("Excluded frames:", np.sum(~mask_excl))
print("Remaining frames:", np.sum(mask_excl))
print("Expected excluded:", exclude_end - exclude_start + 1)

Total frames: 1798
Excluded frames: 14
Remaining frames: 1784
Expected excluded: 14


In [11]:
bin_edges = np.arange(0.0, 0.45 + 0.05, 0.05)  # 0.00 to 0.45
bin_labels = [f"{bin_edges[i]:.2f}–{bin_edges[i+1]:.2f}" for i in range(len(bin_edges)-1)]

def count_bins(speed, mask=None):
    if mask is None:
        mask = np.ones_like(speed, dtype=bool)
    idx = np.digitize(speed[mask], bin_edges, right=False) - 1
    valid = (idx >= 0) & (idx < len(bin_labels))
    counts = np.bincount(idx[valid], minlength=len(bin_labels))
    return counts

counts_all = count_bins(gt_speed)
counts_excl = count_bins(gt_speed, mask_excl)

print("GT speed bins:")
for label, a, b in zip(bin_labels, counts_all, counts_excl):
    print(f"{label:>10} | all={a:4d} | excl={b:4d}")

GT speed bins:
 0.00–0.05 | all=  46 | excl=  46
 0.05–0.10 | all= 144 | excl= 144
 0.10–0.15 | all=  74 | excl=  74
 0.15–0.20 | all= 216 | excl= 202
 0.20–0.25 | all= 476 | excl= 476
 0.25–0.30 | all= 517 | excl= 517
 0.30–0.35 | all= 278 | excl= 278
 0.35–0.40 | all=  44 | excl=  44
 0.40–0.45 | all=   3 | excl=   3


In [12]:
def summarize_bins(speed, err_x, err_y, err_abs, mask=None):
    if mask is None:
        mask = np.ones_like(speed, dtype=bool)

    idx = np.digitize(speed[mask], bin_edges, right=False) - 1
    valid = (idx >= 0) & (idx < len(bin_labels))

    s = speed[mask][valid]
    ex = err_x[mask][valid]
    ey = err_y[mask][valid]
    ea = err_abs[mask][valid]
    idx = idx[valid]

    rows = []
    for i, label in enumerate(bin_labels):
        m = idx == i
        if not np.any(m):
            continue
        rows.append({
            "bin": label,
            "n": int(np.sum(m)),
            "forward_median": float(np.median(ex[m])),
            "lateral_median": float(np.median(ey[m])),
            "absolute_median": float(np.median(ea[m])),
            "forward_mean": float(np.mean(ex[m])),
            "lateral_mean": float(np.mean(ey[m])),
            "absolute_mean": float(np.mean(ea[m])),
        })
    return rows

summary_all = summarize_bins(gt_speed, err_x, err_y, err_abs)
summary_excl = summarize_bins(gt_speed, err_x, err_y, err_abs, mask_excl)

print("ALL FRAMES")
for r in summary_all:
    print(r)

print("\nEXCLUDING 546-559")
for r in summary_excl:
    print(r)

ALL FRAMES
{'bin': '0.00–0.05', 'n': 46, 'forward_median': 0.006412727159667937, 'lateral_median': -0.0034422985959459383, 'absolute_median': 0.03103608036014871, 'forward_mean': 0.023909439471310176, 'lateral_mean': -0.002540323116821613, 'absolute_mean': 0.048176848313555204}
{'bin': '0.05–0.10', 'n': 144, 'forward_median': -0.009865782433483032, 'lateral_median': -0.029338439202921094, 'absolute_median': 0.05097795666689211, 'forward_mean': -0.0016565393744470626, 'lateral_mean': -0.02251863283766574, 'absolute_mean': 0.06160879979371187}
{'bin': '0.10–0.15', 'n': 74, 'forward_median': 0.031430138855793974, 'lateral_median': -0.00975487655726725, 'absolute_median': 0.08162589654823532, 'forward_mean': 0.04015122153712185, 'lateral_mean': -0.013633627658959183, 'absolute_mean': 0.08310794751146179}
{'bin': '0.15–0.20', 'n': 216, 'forward_median': 0.07455648721884034, 'lateral_median': 0.004321679103110374, 'absolute_median': 0.0811674063477563, 'forward_mean': 0.08039243866908986, 'l